In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

In [40]:
import pandas as pd
data_new_worth = pd.read_csv("/content/net_worth_cleaned.csv", encoding='ISO-8859-1', index_col='observation_date', parse_dates=True)
data_unemployment = pd.read_csv("/content/unemployment_cleaned.csv", encoding='ISO-8859-1', index_col='observation_date', parse_dates=True)

In [44]:
data_new_worth, data_unemployment = data_new_worth.align(data_unemployment, join='inner', axis=0)
data_new_worth = data_new_worth.asfreq('QS')
data_unemployment = data_unemployment.asfreq('QS')
last_year = data_new_worth.index.year.max()
train = data_new_worth.index.year < last_year
test  = data_new_worth.index.year == last_year

unemployment_train = data_unemployment[train]
unemployment_test  = data_unemployment[test]

train_all = data_new_worth[train]
test_all  = data_new_worth[test]

In [45]:
results = []
for y_col in data_new_worth.columns:
    y_train = train_all[y_col]
    y_test  = test_all[y_col]
    model = LinearRegression()
    model.fit(unemployment_train, y_train)
    y_pred = model.predict(unemployment_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    percent_error = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    results.append({
        'Net Worth Group': y_col,
        'RMSE': rmse,
        'R^2': r2,
        'Percent Error (%)': percent_error
    })

In [46]:
results_df = pd.DataFrame(results)
print(results_df)

  Net Worth Group          RMSE         R^2  Percent Error (%)
0   99.9% to 100%  1.354903e+07 -132.151692          46.781826
1    99% to 99.9%  2.462370e+06 -497.262308          59.391242
2     99% to 100%  2.773789e+07 -200.387380          45.162970
3      90% to 99%  2.336207e+07 -565.243052          46.667026
4      50% to 90%  2.636364e+07 -131.574719          49.959965
5       0% to 50%  1.281595e+07 -130.929682          53.817736


In [47]:
train = (data_new_worth.index.year >= 2004) & (data_new_worth.index.year <= 2024)
test = (data_new_worth.index.year >= 2025)

y_train_all = data_new_worth[train]
y_test_all  = data_new_worth[test]

X_train_all = data_unemployment[train]
X_test_all  = data_unemployment[test]

In [48]:
model_fits = {}

for y_col in y_train_all.columns:
    for x_col in X_train_all.columns:
        y_train = y_train_all[y_col].dropna()
        X_train = X_train_all[[x_col]].loc[y_train.index]
        try:
            model = ARIMA(y_train, exog=X_train, order=(1,1,1))
            model_fit = model.fit()
            model_fits[(y_col, x_col)] = model_fit
            print(f"Fitted: {y_col} ~ {x_col}")
        except Exception as e:
            print(f"Skipped {y_col} ~ {x_col}: {e}")

Fitted: 99.9% to 100% ~ 20-24
Fitted: 99.9% to 100% ~ 25-34
Fitted: 99.9% to 100% ~ 35-44
Fitted: 99.9% to 100% ~ 45-54
Fitted: 99.9% to 100% ~ 55 & Over
Fitted: 99% to 99.9% ~ 20-24
Fitted: 99% to 99.9% ~ 25-34
Fitted: 99% to 99.9% ~ 35-44
Fitted: 99% to 99.9% ~ 45-54
Fitted: 99% to 99.9% ~ 55 & Over
Fitted: 99% to 100% ~ 20-24
Fitted: 99% to 100% ~ 25-34
Fitted: 99% to 100% ~ 35-44
Fitted: 99% to 100% ~ 45-54
Fitted: 99% to 100% ~ 55 & Over
Fitted: 90% to 99% ~ 20-24
Fitted: 90% to 99% ~ 25-34
Fitted: 90% to 99% ~ 35-44
Fitted: 90% to 99% ~ 45-54
Fitted: 90% to 99% ~ 55 & Over
Fitted: 50% to 90% ~ 20-24
Fitted: 50% to 90% ~ 25-34
Fitted: 50% to 90% ~ 35-44
Fitted: 50% to 90% ~ 45-54
Fitted: 50% to 90% ~ 55 & Over
Fitted: 0% to 50% ~ 20-24
Fitted: 0% to 50% ~ 25-34
Fitted: 0% to 50% ~ 35-44
Fitted: 0% to 50% ~ 45-54
Fitted: 0% to 50% ~ 55 & Over


In [49]:
predictions = {}
for (y_col, x_col), model_fit in model_fits.items():
    X_test = X_test_all[[x_col]]
    try:
        forecast = model_fit.forecast(steps=len(X_test), exog=X_test)
        predictions[(y_col, x_col)] = forecast
    except Exception as e:
        print(f"Forecast failed for {y_col} ~ {x_col}: {e}")

In [51]:
rows = []

for (y_col, x_col), pred in predictions.items():
    for date, value in zip(pred.index, pred.values):
        rows.append({
            'Date': date,
            'Net Worth (y)': y_col,
            'Unemployment (x)': x_col,
            'Forecast': value,
            'Actual': y_test_all.loc[date, y_col]
        })

forecast_df = pd.DataFrame(rows)

print(forecast_df)

         Date  Net Worth (y) Unemployment (x)      Forecast    Actual
0  2025-01-01  99.9% to 100%            20-24  2.758114e+07  27062861
1  2025-04-01  99.9% to 100%            20-24  2.780479e+07  28519246
2  2025-07-01  99.9% to 100%            20-24  2.803077e+07  29938926
3  2025-01-01  99.9% to 100%            25-34  2.757694e+07  27062861
4  2025-04-01  99.9% to 100%            25-34  2.779559e+07  28519246
..        ...            ...              ...           ...       ...
85 2025-04-01      0% to 50%            45-54  2.290177e+07  23464241
86 2025-07-01      0% to 50%            45-54  2.310290e+07  24887247
87 2025-01-01      0% to 50%        55 & Over  2.268667e+07  22154938
88 2025-04-01      0% to 50%        55 & Over  2.289337e+07  23464241
89 2025-07-01      0% to 50%        55 & Over  2.309761e+07  24887247

[90 rows x 5 columns]


In [50]:
results = []
for (y_col, x_col), pred in predictions.items():
    actual = y_test_all[y_col].loc[pred.index]
    rmse = np.sqrt(mean_squared_error(actual, pred))
    r2 = r2_score(actual, pred)
    percent_error = np.mean(np.abs((actual - pred) / actual)) * 100
    results.append({
        'Net Worth (y)': y_col,
        'Unemployment (x)': x_col,
        'RMSE': rmse,
        'R^2': r2,
        'Percent Error (%)': percent_error
    })
results_df = pd.DataFrame(results)
print(results_df)

    Net Worth (y) Unemployment (x)          RMSE       R^2  Percent Error (%)
0   99.9% to 100%            20-24  1.213824e+06 -0.068665           3.597918
1   99.9% to 100%            25-34  1.216516e+06 -0.073410           3.606624
2   99.9% to 100%            35-44  1.220496e+06 -0.080446           3.608886
3   99.9% to 100%            45-54  1.212769e+06 -0.066808           3.593165
4   99.9% to 100%        55 & Over  1.217381e+06 -0.074937           3.603980
5    99% to 99.9%            20-24  1.284647e+05 -0.356184           2.653091
6    99% to 99.9%            25-34  1.287102e+05 -0.361373           2.656680
7    99% to 99.9%            35-44  1.287550e+05 -0.362321           2.654646
8    99% to 99.9%            45-54  1.285340e+05 -0.357649           2.654174
9    99% to 99.9%        55 & Over  1.283724e+05 -0.354238           2.650433
10    99% to 100%            20-24  1.918949e+06  0.036144           2.732113
11    99% to 100%            25-34  1.920872e+06  0.034211      